# Custom Causal Graph Data Simulation

This notebook generates simulated data based on our manually designed causal graph for credit scoring.

The original data generation framework uses randomly generated ER or BA/SF graphs. In this notebook, we do not use a random graph structure. Instead, we manually define a domain-specific DAG and generate data according to this custom causal structure.

This notebook includes:

1. Manually defining the causal graph structure
2. Manually defining the ground-truth weighted adjacency matrix
3. Generating data based on structural equations
4. Distinguishing between full data and observed data
5. Saving the simulated data and adjacency matrices

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from IPython.display import display

# 1. Define Variables

According to our designed causal graph, the variables are:

- Maternity_Leave: whether the individual has maternity-leave-related experience; observed variable
- Risk_Aversion: degree of risk aversion; latent variable
- Family_Labor: degree of family labor or family-focused responsibility; latent variable
- Length_Credit_History: length of credit history; observed variable
- Default_Records: default history; observed variable
- Credit_Mix: credit mix indicator; observed variable
- Unemployment: unemployment status; observed variable
- Credit_Score: credit score; outcome variable

Risk_Aversion and Family_Labor are unobserved variables. They exist in the true data-generating process, but they will be removed from the final observed dataset.

In [ ]:
nodes = [
    "Maternity_Leave",
    "Risk_Aversion",
    "Family_Labor",
    "Length_Credit_History",
    "Default_Records",
    "Credit_Mix",
    "Unemployment",
    "Credit_Score"
]

latent_nodes = [
    "Risk_Aversion",
    "Family_Labor"
]

observed_nodes = [
    node for node in nodes if node not in latent_nodes
]

idx = {
    node: i for i, node in enumerate(nodes)
}

# 2. Define the Custom DAG

Here we do not use an ER or BA/SF random graph. Instead, we manually specify the causal edges.

The main causal paths are:

- Maternity_Leave → Risk_Aversion
- Maternity_Leave → Family_Labor
- Risk_Aversion → Length_Credit_History
- Risk_Aversion → Default_Records
- Family_Labor → Credit_Mix
- Family_Labor → Unemployment
- Length_Credit_History → Credit_Score
- Default_Records → Credit_Score
- Credit_Mix → Credit_Score
- Unemployment → Credit_Score

The graph must be a DAG, meaning a directed acyclic graph.

In [ ]:
edges = [
    ("Maternity_Leave", "Risk_Aversion"),
    ("Maternity_Leave", "Family_Labor"),

    ("Risk_Aversion", "Length_Credit_History"),
    ("Risk_Aversion", "Default_Records"),

    ("Family_Labor", "Credit_Mix"),
    ("Family_Labor", "Unemployment"),

    ("Length_Credit_History", "Credit_Score"),
    ("Default_Records", "Credit_Score"),
    ("Credit_Mix", "Credit_Score"),
    ("Unemployment", "Credit_Score")
]

G = nx.DiGraph()
G.add_nodes_from(nodes)
G.add_edges_from(edges)

assert nx.is_directed_acyclic_graph(G)

full_adjacency = nx.to_pandas_adjacency(G, nodelist=nodes, dtype=int)

display(full_adjacency)

# 3. Visualize the Custom DAG

The following figure visualizes the manually defined causal graph.

The direction of each arrow represents the assumed causal direction:

Parent node → Child node

In [ ]:
plt.figure(figsize=(14, 7))

pos = {
    "Maternity_Leave": (0, 0),
    "Risk_Aversion": (1.8, 1),
    "Family_Labor": (1.8, -1),
    "Length_Credit_History": (4.2, 1.4),
    "Default_Records": (4.2, 0.4),
    "Credit_Mix": (4.2, -0.6),
    "Unemployment": (4.2, -1.6),
    "Credit_Score": (6.5, 0)
}

nx.draw(
    G,
    pos,
    with_labels=True,
    node_size=3200,
    font_size=9,
    arrows=True,
    arrowsize=18
)

plt.axis("off")
plt.show()

# 4. Define the Weighted Adjacency Matrix

Next, we define the ground-truth weighted adjacency matrix W.

In matrix W:

\[
W[i, j] \neq 0
\]

means:

\[
X_i \rightarrow X_j
\]

In other words, parent variable \(X_i\) has a causal effect on child variable \(X_j\).

The sign of each weight has the following interpretation:

- Positive weight: an increase in the parent variable tends to increase the child variable
- Negative weight: an increase in the parent variable tends to decrease the child variable

These weights are the ground-truth causal effects used for data simulation. They are not learned by a model.

In [ ]:
weights = {
    ("Maternity_Leave", "Risk_Aversion"): 1.2,
    ("Maternity_Leave", "Family_Labor"): 1.5,

    ("Risk_Aversion", "Length_Credit_History"): -0.5,
    ("Risk_Aversion", "Default_Records"): -0.25,

    ("Family_Labor", "Credit_Mix"): -0.5,
    ("Family_Labor", "Unemployment"): 0.45,

    ("Length_Credit_History", "Credit_Score"): 8.0,
    ("Default_Records", "Credit_Score"): -65.0,
    ("Credit_Mix", "Credit_Score"): 30.0,
    ("Unemployment", "Credit_Score"): -45.0
}

W = pd.DataFrame(
    data=np.zeros((len(nodes), len(nodes))),
    index=nodes,
    columns=nodes
)

for (parent, child), beta in weights.items():
    W.loc[parent, child] = beta

display(W)

# 5. Variable Types

Not all variables in this simulation are continuous.

The variable types are defined as follows:

| Variable | Type | Description |
|---|---|---|
| Maternity_Leave | Binary | 0 or 1 |
| Risk_Aversion | Ordinal | Score from 1 to 10 |
| Family_Labor | Ordinal | Score from 1 to 10 |
| Length_Credit_History | Continuous | Length of credit history |
| Default_Records | Categorical | 0 = Never, 1 = 0-3 times, 2 = Multiple times |
| Credit_Mix | Binary | 0 or 1 |
| Unemployment | Binary | 0 or 1 |
| Credit_Score | Continuous | Credit score, approximately from 300 to 850 |

In [ ]:
variable_types = pd.DataFrame({
    "Variable": nodes,
    "Type": [
        "Binary",
        "Ordinal",
        "Ordinal",
        "Continuous",
        "Categorical",
        "Binary",
        "Binary",
        "Continuous"
    ],
    "Observed": [
        True,
        False,
        False,
        True,
        True,
        True,
        True,
        True
    ]
})

display(variable_types)

# 6. Define the Data Simulation Function

The following function generates simulated data according to the custom causal graph.

The full dataset, full_df, contains all variables, including the latent variables:

- Risk_Aversion
- Family_Labor

The observed dataset, observed_df, removes the latent variables and only keeps variables that are assumed to be observable.

This reflects a realistic setting where some variables exist in the true causal system but are not directly observed by the researcher or by the prediction model.

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def simulate_credit_graph(n=1000, seed=0):
    rng = np.random.default_rng(seed)

    maternity_leave = rng.binomial(1, 0.35, size=n)

    risk_aversion_continuous = (
        5
        + W.loc["Maternity_Leave", "Risk_Aversion"] * maternity_leave
        + rng.normal(0, 1.2, size=n)
    )

    risk_aversion = np.clip(
        np.rint(risk_aversion_continuous),
        1,
        10
    ).astype(int)

    family_labor_continuous = (
        5
        + W.loc["Maternity_Leave", "Family_Labor"] * maternity_leave
        + rng.normal(0, 1.2, size=n)
    )

    family_labor = np.clip(
        np.rint(family_labor_continuous),
        1,
        10
    ).astype(int)

    length_credit_history = (
        12
        + W.loc["Risk_Aversion", "Length_Credit_History"] * risk_aversion
        + rng.normal(0, 2.0, size=n)
    )

    length_credit_history = np.clip(
        length_credit_history,
        0,
        40
    )

    default_latent_score = (
        1.6
        + W.loc["Risk_Aversion", "Default_Records"] * risk_aversion
        + rng.normal(0, 0.9, size=n)
    )

    default_records = np.digitize(
        default_latent_score,
        bins=[0.5, 1.5]
    )

    credit_mix_probability = sigmoid(
        2.5
        + W.loc["Family_Labor", "Credit_Mix"] * family_labor
    )

    credit_mix = rng.binomial(
        1,
        credit_mix_probability,
        size=n
    )

    unemployment_probability = sigmoid(
        -3.0
        + W.loc["Family_Labor", "Unemployment"] * family_labor
    )

    unemployment = rng.binomial(
        1,
        unemployment_probability,
        size=n
    )

    credit_score = (
        650
        + W.loc["Length_Credit_History", "Credit_Score"] * length_credit_history
        + W.loc["Default_Records", "Credit_Score"] * default_records
        + W.loc["Credit_Mix", "Credit_Score"] * credit_mix
        + W.loc["Unemployment", "Credit_Score"] * unemployment
        + rng.normal(0, 30, size=n)
    )

    credit_score = np.clip(
        credit_score,
        300,
        850
    )

    full_df = pd.DataFrame({
        "Maternity_Leave": maternity_leave,
        "Risk_Aversion": risk_aversion,
        "Family_Labor": family_labor,
        "Length_Credit_History": length_credit_history,
        "Default_Records": default_records,
        "Credit_Mix": credit_mix,
        "Unemployment": unemployment,
        "Credit_Score": credit_score
    })

    observed_df = full_df.drop(
        columns=latent_nodes
    )

    return full_df, observed_df


# 7. Generate Data

Now we generate the simulated dataset.

In this example, we use:

- Sample size: n = 1000
- Random seed: seed = 1

full_df is the complete dataset, including latent variables.

observed_df is the observed dataset, excluding latent variables.

In [ ]:
full_df, observed_df = simulate_credit_graph(
    n=1000,
    seed=1
)

display(full_df.head())
display(observed_df.head())

# 8. Check Summary Statistics

We check the basic summary statistics of the generated data to verify that the variable ranges and distributions are reasonable.

In [ ]:
display(full_df.describe().T)

# 9. Check Discrete Variable Distributions

We examine the distributions of binary, ordinal, and categorical variables.

These variables include:

- Maternity_Leave
- Risk_Aversion
- Family_Labor
- Default_Records
- Credit_Mix
- Unemployment

In [ ]:
discrete_columns = [
    "Maternity_Leave",
    "Risk_Aversion",
    "Family_Labor",
    "Default_Records",
    "Credit_Mix",
    "Unemployment"
]

for column in discrete_columns:
    print(column)
    display(full_df[column].value_counts().sort_index())
    print()

# 10. Observed Data and Hidden Variables

In the true causal graph, Risk_Aversion and Family_Labor affect later variables.

However, these two variables are removed from observed_df.

This means that the observed dataset contains unobserved-variable bias.

If a causal discovery algorithm is later applied to observed_df, the algorithm can only observe:

- Maternity_Leave
- Length_Credit_History
- Default_Records
- Credit_Mix
- Unemployment
- Credit_Score

It cannot observe:

- Risk_Aversion
- Family_Labor

Therefore, the discovered causal structure may be biased or incomplete.

In [ ]:
display(observed_df.head())

# 11. Create Observed Direct Adjacency Matrix

Here we create two adjacency matrices:

1. full_adjacency: the complete true causal graph, including latent variables
2. observed_direct_adjacency: the direct causal graph among observed variables only

Important note: observed_direct_adjacency only represents direct causal edges among observed variables. It does not represent indirect effects or confounding effects caused by latent variables.

In [ ]:
observed_direct_adjacency = full_adjacency.loc[
    observed_nodes,
    observed_nodes
]

observed_direct_weighted_adjacency = W.loc[
    observed_nodes,
    observed_nodes
]

display(full_adjacency)
display(observed_direct_adjacency)
display(observed_direct_weighted_adjacency)

# 12. Save Data

We save the generated datasets and ground-truth causal matrices.

The output files include:

- custom_credit_full_data.csv
- custom_credit_observed_data.csv
- custom_credit_full_adjacency.csv
- custom_credit_full_weighted_adjacency.csv
- custom_credit_observed_direct_adjacency.csv
- custom_credit_observed_direct_weighted_adjacency.csv

In [ ]:
full_df.to_csv(
    "custom_credit_full_data.csv",
    index=False
)

observed_df.to_csv(
    "custom_credit_observed_data.csv",
    index=False
)

full_adjacency.to_csv(
    "custom_credit_full_adjacency.csv"
)

W.to_csv(
    "custom_credit_full_weighted_adjacency.csv"
)

observed_direct_adjacency.to_csv(
    "custom_credit_observed_direct_adjacency.csv"
)

observed_direct_weighted_adjacency.to_csv(
    "custom_credit_observed_direct_weighted_adjacency.csv"
)

# 13. Final Output

At this point, we have completed the data simulation based on the custom causal graph.

The following objects can be used for later analysis:

- observed_df: input data for causal discovery algorithms
- full_adjacency: complete ground-truth causal graph
- W: complete ground-truth weighted causal graph
- observed_direct_adjacency: direct ground-truth causal graph among observed variables only

It is important to note that Risk_Aversion and Family_Labor are latent variables. Therefore, observed_df does not satisfy the causal sufficiency assumption. As a result, the causal discovery results based on observed_df may not fully match the complete true causal graph.

In [ ]:
print("Full data shape:", full_df.shape)
print("Observed data shape:", observed_df.shape)
print("Full adjacency shape:", full_adjacency.shape)
print("Observed direct adjacency shape:", observed_direct_adjacency.shape)